# Processing MESACLIP data - sea ice monthly, single ensembles

* **Description**: Reads in MESACLIP historical and crops to just the Arctic
* **Creator**: Alice DuVivier
* **Date**: December 2025

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from glob import glob
from datetime import datetime
import xesmf as xe
import matplotlib.pyplot as plt
import scipy.sparse as sps

import dask
from distributed import Client
from ncar_jobqueue import NCARCluster

## Spin up DASK cluster

In [2]:
# Create our NCAR Cluster - which uses PBSCluster under the hood
cluster = NCARCluster(walltime='2:00:00', cores=1, processes=1, memory='100 GB', 
                      resource_spec='select=1:ncpus=1:mem=100GB',interface='ext')

# Spin up 32 workers
cluster.scale(32)

# Assign the cluster to our Client
client = Client(cluster)

/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:274: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/con

In [3]:
client
# cluster.close()

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.176:35069,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Set MESACLIP ensemble information

In [4]:
# Historical
path_base = '/gdex/data/d651007/'
case_base = 'b.e13.BHISTC5.ne120_t12.cesm-ihesp-'
# list 10 members of MESACLIP - last one is the first and named differently
ens_mems = ['sehires38-1850-2005.001', 'hires1.0.30-1920-2005.002', 
            'hires1.0.30-1920-2005.003', 'hires1.0.44-1920-2005.004', 
            'hires1.0.44-1920-2005.005', 'hires1.0.45-1920-2005.006', 
            'hires1.0.46-1920-2005.007', 'hires1.0.46-1920-2005.008',
            'hires1.0.46-1920-2005.009', 'hires1.0.46-1920-2005.010']

In [5]:
# set the groups of years since we have to load them separately (UGH)
# 1920s, 1930s, 1940s, 1950s, 1960s, 1970s, 1980s, 1990s, 2000s
year_sets = ['192','193','194','195','196',
             '197','198','199','200']

## Load in SEA ICE Data

In [6]:
# Choose variable to load
# 'aice', 'hi'
var_in = 'aice'

In [166]:
# Choose ensemble and years to load
n = 1  # 0-9
y = 5  # 0-8

ens_in = ens_mems[n]
print(ens_in)
yr_pre = year_sets[y]
print(yr_pre)

print('loading ensemble:'+ens_in+' for years '+yr_pre+'0s')

hires1.0.30-1920-2005.002
197
loading ensemble:hires1.0.30-1920-2005.002 for years 1970s


In [167]:
%%time

print('loading historical data for member: '+ens_in)

# set case
case = case_base + ens_in
# set data directory
path = path_base+case+'/ice/proc/tseries/month_1'
# get list of files
files = sorted(glob(f'{path}/{case}.cice.h.{var_in}.{yr_pre}???-??????.nc'))       
# load files
ds_tmp = xr.open_mfdataset(files, data_vars="minimal", coords='minimal', compat="override", parallel=True, 
                           concat_dim="time", combine='nested', decode_times=True)
# take an average over the time bounds to get the right time dimension
ds_tmp["time"] = ds_tmp.time_bounds.compute().mean(dim="d2")


loading historical data for member: hires1.0.30-1920-2005.002
CPU times: user 91.1 ms, sys: 395 µs, total: 91.5 ms
Wall time: 339 ms


In [168]:
# check data
#ds_tmp.aice.isel(time=10).plot()

## Subset file to Arctic Only
- ">50N"

In [169]:
#ds_tmp.TLAT.plot()

In [170]:
ds_sub = ds_tmp.isel(nj=slice(1750,2400))

In [171]:
# check data
#ds_sub.TLAT.plot()

## Export and save the file
- NOTE: YOU MUST MANUALLY ENTER THE VARIABLE NAME BELOW

In [172]:
# set output path and names
path_out = '/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/mesaclip_all_arctic/monthly/'

# set output file name
nn = n+1
if nn < 10:
    ensnum = '00'+str(nn)
elif nn >= 10:
    ensnum = '010'

file_out = case_base+'hires1.0.30-'+ensnum+'_'+yr_pre+'0-'+yr_pre+'9'+'_'+var_in+'_arctic_monthly.nc'
fout = path_out + file_out
print(fout)

/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/mesaclip_all_arctic/monthly/b.e13.BHISTC5.ne120_t12.cesm-ihesp-hires1.0.30-002_1970-1979_aice_arctic_monthly.nc


In [173]:
ds_out = xr.Dataset()

ds_out['aice'] = ds_sub.aice

# change the attributes
ds_out.attrs['author'] = 'Alice DuVivier'
ds_out.attrs['date_processed'] = datetime.now().strftime('%Y-%m-%d')
ds_out.attrs['contents'] = f'Subset of historical MESACLIP Data'

ds_out

<xarray.Dataset>
Dimensions:  (nj: 650, ni: 3600, time: 120)
Coordinates:
    TLON     (nj, ni) float32 dask.array<chunksize=(650, 1200), meta=np.ndarray>
    TLAT     (nj, ni) float32 dask.array<chunksize=(650, 1200), meta=np.ndarray>
    ULON     (nj, ni) float32 dask.array<chunksize=(650, 1200), meta=np.ndarray>
    ULAT     (nj, ni) float32 dask.array<chunksize=(650, 1200), meta=np.ndarray>
  * time     (time) object 1970-01-16 12:00:00 ... 1979-12-16 12:00:00
Dimensions without coordinates: nj, ni
Data variables:
    aice     (time, nj, ni) float32 dask.array<chunksize=(1, 650, 1200), meta=np.ndarray>
Attributes:
    author:          Alice DuVivier
    date_processed:  2025-12-12
    contents:        Subset of historical MESACLIP Data

In [174]:
# calculate the size of the dataset in GB
size_gb = ds_out.nbytes / (1024**3)
print(f"The dataset is approximately {size_gb:.2f} GB.")

The dataset is approximately 1.08 GB.


In [175]:
# Export the dataset to NetCDF with all attributes and coordinates
ds_out.to_netcdf(fout)

In [176]:
#cluster.close()